# Experimento final: `recortes_negativo_r2_fechamento`, treino com as 37 imagens e avaliação única no teste

**Quando usar:** depois de escolher, **só pela validação cruzada**, a melhor configuração entre os notebooks de experimento. Este
notebook treina essa configuração com as 37 imagens de treino, pelo número de épocas registrado na validação cruzada dela
(mediana das melhores épocas dos *folds*), e avalia as 14 imagens de teste **uma única vez**.

**Como preparar:**
1. Em `EXPERIMENTO_ESCOLHIDO`, ponha o nome do experimento (a pasta em `docs/estudo/resultados/`).
2. Copie do notebook do experimento a célula de configuração inteira (imports, parâmetros e `DESCRIPTION`) e a célula da
   `factory` (a perda e, nos experimentos com recortes, o `TiledPipeline`). A célula de conferência compara com o `config.json`
   gravado e recusa seguir se algo for diferente.

**Saída:** `docs/estudo/resultados/final_<experimento>/` (configuração, métricas por imagem do teste, resumo pareado com o
Cellpose e figura com imagens do teste) e o modelo final em `MyDrive/cell-fuzzy-models/<experimento>/final.pt` (no Colab).

**Ressalva para o texto:** as 14 imagens de teste já foram usadas como validação nos experimentos 1–6 e tiveram o Dice calculado
nos experimentos `legado/`. Daqui em diante, elas só são avaliadas aqui.

**Configuração escolhida (pela validação cruzada):** `recortes_negativo_r2_fechamento`: recortes de 256 px, ScribblePrompt em 128², negativo do
Cellpose com faixa livre de 2 px, fechamento de 3 px do marcador antes do ScribblePrompt (no treino e na avaliação) e só o soft
Dice. Na validação cruzada: 0,8383 (sobreposição 1/4, sem TTA) e 0,8402 com sobreposição 1/2 + TTA; Cellpose 0,8450. Treino final
por 67 épocas (mediana das melhores épocas dos *folds*).

**Números do teste, declarados antes de rodar:** dois, ambos com configurações escolhidas só pela validação:
1. **registro:** a avaliação padrão (blocos com sobreposição 1/4, sem TTA), a mesma da seleção entre experimentos;
2. **melhor inferência:** sobreposição 1/2 + TTA (escolhidas nos testes depois do treino, só nas 37 imagens de treino). **É o
   número principal do trabalho.**

A seção 5b também mede a editabilidade do marcador no teste.

## 0. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

## 1. Configuração (copiada do experimento escolhido)

In [ ]:
import dataclasses
import json

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.data.load.preprocessed_dataset import NEGATIVE_KEY, load_preprocessed
from src.evaluation.figures import pick_cases, prediction_figure
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import save_run, summarize
from src.losses.loss_composer import LossComposer
from src.losses.terms import DiceTerm
from src.models.configs.marker_unet_config import UNET_CONFIG
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.models.networks.marker_unet import MarkerUNet
from src.pipeline.steps.inference.frozen_segmentation_step import FrozenSegmentationStep
from src.pipeline.steps.inference.marker_closing_step import MarkerClosingStep
from src.pipeline.steps.inference.marker_step import MarkerStep
from src.pipeline.training_pipeline import TrainingPipeline
from src.pipeline.tiled_pipeline import TiledPipeline
from src.training.kfold import TrainConfig, run_final

# Nome do experimento escolhido pela validação cruzada (a pasta em docs/estudo/resultados/). A configuração
# abaixo é a dele, copiada do notebook do experimento; a célula seguinte confere.
EXPERIMENTO_ESCOLHIDO = "recortes_negativo_r2_fechamento"
EXPERIMENT_NAME = f"final_{EXPERIMENTO_ESCOLHIDO}"
OVERWRITE = False            # recusa gravar por cima de resultados ou modelos que já existem

RGBA_ALPHA = "mask"          # 4º canal da MarkerUNet: máscara do Cellpose ("prob": probabilidade)
TILE_SIZE = 256             # recorte do treino e bloco da avaliação (px da imagem original)
CROPS_PER_IMAGE = 4          # recortes sorteados de cada imagem por época
TILE_OVERLAP = 0.25          # sobreposição dos blocos na avaliação (fração do lado)
SP_INPUT_SIZE = (128, 128)   # entrada do ScribblePrompt (o recorte reduzido)
TAU = 0.5                    # limiar do canal positivo
RAIO_FECHAMENTO = 3          # fechamento do marcador antes do ScribblePrompt, em px (0: sem fechamento)
NEGATIVE_RADIUS = 2              # None: sem negativo; n: negativo = pixels a mais de n px de qualquer máscara do Cellpose

CONFIG = TrainConfig(
    k=5, batch_size=16, max_epochs=200, patience=20, min_delta=0.0,
    lr=1e-4, grad_clip=1.0, threshold=0.5, seed=42, log_every=10,
    crop_size=TILE_SIZE, crops_per_image=CROPS_PER_IMAGE,
)
DESCRIPTION = {
    "perdas": "soft Dice (1,0), sem regularização do marcador",
    "rede_final": f"ScribblePrompt-UNet v1 congelado, scribble_mode=positive, tau={TAU}, input_size={SP_INPUT_SIZE}"
                  + ("" if NEGATIVE_RADIUS is None
                     else f", negativo = fundo do Cellpose (faixa livre de {NEGATIVE_RADIUS} px)"),
    "fechamento": f"fechamento em tons de cinza do marcador por um disco de {RAIO_FECHAMENTO} px "
                  "antes do ScribblePrompt, no treino e na avaliação",
    "marker_net": "MarkerUNet (smp Unet, resnet34 ImageNet, 4 canais)",
    "rgba_alpha": RGBA_ALPHA,
    "aumentacao": "rot90 + flips, sorteada a cada época",
    "recortes": f"treino com {CROPS_PER_IMAGE} recortes de {TILE_SIZE}² por imagem por época (MarkerUNet sem redução); avaliação da imagem inteira em blocos de {TILE_SIZE}² com sobreposição {TILE_OVERLAP} e janela de Hann",
}

assert torch.cuda.is_available() or "google.colab" not in sys.modules, "No Colab, use um ambiente com GPU."
print("Device:", CONFIG.device)
print(json.dumps(DESCRIPTION, indent=2, ensure_ascii=False))

Conferência com a configuração gravada na validação cruzada do experimento escolhido:

In [ ]:
if EXPERIMENTO_ESCOLHIDO is None:
    raise ValueError("Defina EXPERIMENTO_ESCOLHIDO com o nome do experimento escolhido pela validação cruzada.")
with open(os.path.join("docs", "estudo", "resultados", EXPERIMENTO_ESCOLHIDO, "config.json"), encoding="utf-8") as f:
    REFERENCIA = json.load(f)

atual = {k: v for k, v in dataclasses.asdict(CONFIG).items() if k not in ("device", "log_every")}
gravada = {k: v for k, v in REFERENCIA["config"].items() if k not in ("device", "log_every")}
if REFERENCIA["descricao"] != DESCRIPTION or atual != gravada:
    raise ValueError("A configuração deste notebook não é a do experimento escolhido; copie-a do notebook dele.")
N_EPOCHS = REFERENCIA["final_epochs"]
print(f"Experimento {EXPERIMENTO_ESCOLHIDO}: configuração conferida; treino final por {N_EPOCHS} épocas.")

## 2. Onde o modelo final é gravado

In [ ]:
# Modelos treinados: no Colab, no Google Drive de quem roda o notebook (sobrevive ao fim da sessão); fora dele,
# em ~/cell-fuzzy-models. Troque MODELS_ROOT para gravar em outro lugar.
if "google.colab" in sys.modules:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        MODELS_ROOT = "/content/drive/MyDrive/cell-fuzzy-models"
    except Exception as erro:
        print(f"Drive não montado ({erro}): os modelos ficam no disco da sessão e somem quando ela terminar.")
        MODELS_ROOT = "/content/cell-fuzzy-models"
else:
    MODELS_ROOT = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))
MODELS_DIR = os.path.join(MODELS_ROOT, EXPERIMENTO_ESCOLHIDO)
print("Modelos em:", MODELS_DIR)

## 3. Dados

In [ ]:
train_samples = load_preprocessed("train", alpha=RGBA_ALPHA, negative_radius=NEGATIVE_RADIUS)
test_samples = load_preprocessed("test", alpha=RGBA_ALPHA, negative_radius=NEGATIVE_RADIUS)
print(f"Treino: {len(train_samples)} imagens | Teste: {len(test_samples)} imagens")

## 4. Rede final e montagem do experimento

In [ ]:
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_ROOT, "scribbleprompt"))
FINAL_NET = ScribblePromptingNetwork(
    checkpoint=checkpoint, device=CONFIG.device, scribble_mode="positive",
    positive_threshold=TAU, input_size=SP_INPUT_SIZE,
)


def factory():
    """Monta um experimento novo: MarkerUNet recém-inicializada, pipeline em blocos e perdas.

    O treino passa recortes de TILE_SIZE² (a MarkerUNet os vê sem redução); a avaliação passa a imagem
    inteira, que o TiledPipeline divide em blocos de TILE_SIZE² e cola de volta. O ScribblePrompt é
    congelado e não guarda estado, então a mesma instância serve a todos os folds. Entre a MarkerUNet e o
    ScribblePrompt, o marcador passa pelo fechamento de RAIO_FECHAMENTO px (o gradiente atravessa o máximo e o
    mínimo); a chave markers continua sendo o marcador cru.
    """
    marker_net = MarkerUNet(config=UNET_CONFIG)
    pipeline = TrainingPipeline([
        MarkerStep(model=marker_net, differentiable=True, target_size=TILE_SIZE, device=CONFIG.device),
        MarkerClosingStep(radius=RAIO_FECHAMENTO),
        FrozenSegmentationStep(final_network=FINAL_NET, device=CONFIG.device, markers_key="markers_closed",
                               negative_key=None if NEGATIVE_RADIUS is None else NEGATIVE_KEY),
    ])
    composer = LossComposer([DiceTerm()])
    return marker_net, TiledPipeline(pipeline, tile_size=TILE_SIZE, overlap=TILE_OVERLAP), composer

## 5. Treino com as 37 imagens e avaliação única no teste

In [ ]:
test_rows, final_history, final_pipeline = run_final(
    factory, train_samples, test_samples, CONFIG, num_epochs=N_EPOCHS, models_dir=MODELS_DIR, overwrite=OVERWRITE,
)
run_dir = save_run(EXPERIMENT_NAME, CONFIG, [], test_rows=test_rows, final_epochs=N_EPOCHS, description=DESCRIPTION,
                   overwrite=OVERWRITE)
print("Resultados em:", run_dir)

d = summarize(test_rows)["dice"]
print(f"teste ({len(test_rows)} imagens): Dice {d['mean']:.4f} ± {d['std']:.4f} (IC95 {d['ci95'][0]:.4f}–{d['ci95'][1]:.4f}) "
      f"| Cellpose {d['baseline_mean']:.4f} | Δ {d['diff_mean']:+.4f} (IC95 {d['diff_ci95'][0]:+.4f}–{d['diff_ci95'][1]:+.4f}) "
      f"| melhora/piora {d['n_better']}/{d['n_worse']} | Wilcoxon p {d['wilcoxon_p']:.3g}")

### Como ler o resumo acima

- **`Resultados em`:** a pasta onde ficaram as métricas por imagem do teste, a configuração e a figura.
- **Linha `teste`** — as 14 imagens de teste, previstas pelo modelo final (treinado com as 37 imagens de treino):
  - **`Dice média ± desvio`:** o Dice médio das 14 imagens e o desvio padrão entre elas. Dice vai de 0 (nada em comum com o
    ground truth) a 1 (igual ao ground truth).
  - **`IC95`:** intervalo de confiança de 95% da média, por *bootstrap* (reamostragem das imagens).
  - **`Cellpose`:** o Dice médio do Cellpose sozinho nas mesmas 14 imagens.
  - **`Δ`:** a diferença média, imagem por imagem, entre o pipeline e o Cellpose (negativa = o pipeline perde), com o IC95 dela.
  - **`melhora/piora`:** em quantas imagens o pipeline ganhou e em quantas perdeu do Cellpose.
  - **`Wilcoxon p`:** teste pareado; com só 14 imagens, ele precisa de diferenças grandes e consistentes para ficar abaixo de 0,05.

## 5b. Teste com a melhor inferência e editabilidade do marcador

O mesmo modelo final, nas 14 imagens de teste, em duas condições: **registro** (sobreposição 1/4, sem TTA; tem de repetir o Dice
acima) e **melhor inferência** (sobreposição 1/2 + TTA, a média das 8 rotações e espelhamentos), contra o Cellpose, imagem por
imagem. A editabilidade (`src/evaluation/editability.py`) é medida no marcador cru da MarkerUNet (raio 0) e no marcador fechado
que o ScribblePrompt recebe. As tabelas são gravadas em `{run_dir}/teste_melhor_inferencia.csv` e
`{run_dir}/teste_editabilidade.csv`.

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

from src.data.load.preprocessed_dataset import PreprocessedDataset, collate_samples
from src.evaluation.editability import marker_editability
from src.pipeline.steps.inference.marker_closing_step import grey_closing

SOBREPOSICAO_INFERENCIA = 0.5
IMAGENS_20X = {"TCGA-HE-7128-01Z-00-DX1", "TCGA-HE-7129-01Z-00-DX1", "TCGA-HE-7130-01Z-00-DX1"}


def rodar(tiled, batch, tta):
    """Probabilidade (H, W) e marcador cru (H, W); com TTA, a probabilidade é a média das 8 rotações e espelhamentos."""
    with torch.no_grad():
        saida = tiled.run(batch, verbose=False)
        prob, marcador = saida["segmentation"], saida["markers"]
        if tta:
            for k in range(4):
                for espelhar in (False, True):
                    if k == 0 and not espelhar:
                        continue
                    def ida(x):
                        if not (torch.is_tensor(x) and x.ndim == 4):
                            return x
                        x = torch.rot90(x, k, dims=(-2, -1))
                        return torch.flip(x, dims=(-1,)) if espelhar else x
                    s = tiled.run({c: ida(v) for c, v in batch.items()}, verbose=False)["segmentation"]
                    if espelhar:
                        s = torch.flip(s, dims=(-1,))
                    prob = prob + torch.rot90(s, -k, dims=(-2, -1))
            prob = prob / 8
    return prob[0, 0].float().cpu().numpy(), marcador[0, 0].float().cpu().numpy()


CONDICOES = [("registro", TILE_OVERLAP, False), ("melhor inferência", SOBREPOSICAO_INFERENCIA, True)]
LINHAS, MARCADORES = [], {}
for condicao, overlap, tta in CONDICOES:
    tiled = TiledPipeline(final_pipeline.pipeline, tile_size=TILE_SIZE, overlap=overlap)
    for image_id in sorted(test_samples):
        batch = collate_samples([PreprocessedDataset(test_samples, ids=[image_id])[0]])
        batch = {c: (v.to(CONFIG.device) if torch.is_tensor(v) else v) for c, v in batch.items()}
        prob, marcador = rodar(tiled, batch, tta)
        if condicao == "melhor inferência":
            MARCADORES[image_id] = marcador
        s = test_samples[image_id]
        base = binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])
        LINHAS.append({"condicao": condicao, "id": image_id, "aumento_20x": image_id in IMAGENS_20X,
                       **binary_metrics(prob, s["ground_truth"]), **{f"cellpose_{m}": v for m, v in base.items()}})
teste = pd.DataFrame(LINHAS)
teste.to_csv(os.path.join(run_dir, "teste_melhor_inferencia.csv"), index=False)

tabela = []
for condicao in dict.fromkeys(teste.condicao):
    s = teste[teste.condicao == condicao]
    d = s.dice - s.cellpose_dice
    tabela.append({"condição": condicao, "Dice": s.dice.mean(), "precisão": s.precision.mean(),
                   "revocação": s.recall.mean(), "massa": s.mass_ratio.mean(), "Δ Cellpose": d.mean(),
                   "acima/abaixo do Cellpose": f"{int((d > 0).sum())}/{int((d < 0).sum())}",
                   "Wilcoxon p": wilcoxon(s.dice, s.cellpose_dice).pvalue,
                   "Dice 20×": s[s.aumento_20x].dice.mean() if s.aumento_20x.any() else float("nan")})
print(f"Cellpose no teste: Dice {teste[teste.condicao == 'registro'].cellpose_dice.mean():.4f}")
pd.set_option("display.width", 220)
pd.DataFrame(tabela).round(4)

In [ ]:
EDIT = []
for image_id, marcador in MARCADORES.items():
    for r in (0, RAIO_FECHAMENTO):
        fechado = marcador if r == 0 else grey_closing(torch.from_numpy(marcador), r).numpy()
        EDIT.append({"raio": r, "id": image_id, "area_marcador": float((fechado > TAU).mean()),
                     **marker_editability(fechado, test_samples[image_id]["ground_truth"], tau=TAU)})
edit = pd.DataFrame(EDIT)
edit.to_csv(os.path.join(run_dir, "teste_editabilidade.csv"), index=False)
edit.drop(columns=["id"]).groupby("raio").mean().round(3)

### Como ler as tabelas acima

- **Primeira tabela:** uma linha por condição, nas 14 imagens de teste. **registro** tem de repetir o Dice do resumo da seção 5.
  **melhor inferência** é o número principal do trabalho (declarado antes de rodar). **Δ Cellpose, acima/abaixo e Wilcoxon p**
  comparam com o Cellpose imagem por imagem; com 14 imagens, o Wilcoxon só fica abaixo de 0,05 com diferenças grandes e
  consistentes. **Dice 20×:** o teste não tem as três imagens em 20× do treino (fica vazio se nenhuma estiver no teste).
- **Segunda tabela:** a editabilidade do marcador no teste, cru (raio 0) e fechado (3 px). Na validação cruzada, o marcador
  fechado deixou 89% dos núcleos com um único componente.

Para comparar com a validação cruzada: 0,8383 (registro) e 0,8402 (melhor inferência), Cellpose 0,8450 nas 37 imagens de treino;
no teste, o Cellpose dá 0,837.

## 6. Imagens do teste: marcadores e segmentação

In [ ]:
casos = [(rotulo, r, final_pipeline) for rotulo, r in pick_cases(test_rows)]
fig = prediction_figure(casos, test_samples, crop=400, threshold=CONFIG.threshold, device=CONFIG.device)
fig.savefig(os.path.join(run_dir, "figura_teste.png"), dpi=80, bbox_inches="tight")
plt.show()

### Como ler a figura acima

Cada **linha** é uma imagem: as imagens de **teste**, previstas pelo modelo final. As três foram escolhidas pela diferença de Dice contra o Cellpose: a **pior** (onde o
pipeline mais perde), a **mediana** e a **melhor** (onde mais ganha ou menos perde). O rótulo e o nome da imagem estão à esquerda.
Todos os painéis mostram o mesmo recorte central de 400 × 400 px da imagem de 1000 × 1000; os Dice dos títulos são da imagem
inteira.

| Coluna | O que mostra | Como ler |
|---|---|---|
| 1. imagem | a imagem H&E original | os núcleos são as manchas roxas escuras |
| 2. ground truth | a anotação manual: branco = núcleo, preto = fundo | é a resposta certa |
| 3. Cellpose | a segmentação do Cellpose sozinho (branco = núcleo) e o Dice dela | é a linha de base a superar |
| 4. marcador, 0–1 | a saída da MarkerUNet, na escala fixa de 0 (escuro) a 1 (claro); o título dá a fração da imagem com marcador ≥ 0,5 | mostra o nível do marcador; se tudo parecer da mesma cor, ele está quase constante (por exemplo, tudo perto de 0,5) |
| 5. marcador, esticado | o mesmo marcador com as cores esticadas entre os percentis 1 e 99 do recorte (o intervalo está no título) | revela a forma do marcador mesmo quando os valores são parecidos; um bom marcador fica claro sobre os núcleos da coluna 2 e escuro no fundo |
| 6. canal positivo, esticado | o que de fato entra no ScribblePrompt como "aqui há núcleo": `relu(m − 0,5)/0,5`, zero onde o marcador é menor que 0,5; cores esticadas de 0 ao máximo (no título) | um máximo baixo no título significa um positivo fraco; as partes pretas não indicam nada à rede |
| 7. segmentação | a saída final do ScribblePrompt, binarizada no limiar 0,5 (branco = núcleo), e o Dice dela | compare com a coluna 2 (o certo) e com a 3 (o Cellpose): núcleos inchados ou colados indicam excesso; núcleos faltando, falta |

O canal negativo, quando o experimento usa o do Cellpose, não aparece: ele é fixo (1 nos pixels a mais de alguns pixels de
qualquer máscara do Cellpose) e não depende do treino.

## 7. Baixar os resultados (Colab)

In [ ]:
if "google.colab" in sys.modules:
    import shutil

    from google.colab import files

    archive = shutil.make_archive(f"/content/{EXPERIMENT_NAME}", "zip", root_dir=os.path.dirname(run_dir),
                                  base_dir=EXPERIMENT_NAME)
    files.download(archive)  # descompactar em docs/estudo/resultados/